In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
from sqlalchemy.engine import create_engine
from iacele import OdooAPI

load_dotenv(dotenv_path="../.env.local") 

url = os.getenv("API_URL")

api = OdooAPI()

ruta_carpeta = Path.home().joinpath('Documents/Inventario/A1 2025')

Engine para traerte tablas de la database local

In [ ]:
DATABASE_URL = "sqlite:///../iaCeleApp/database/local_iaCele.db"
engine = create_engine(DATABASE_URL)

Ingresar empleados que no están en Odoo

In [ ]:
endpoint = url + '/employees/new'

employees = [
{
  "id_odoo": 1,
  "name": "Emilio Costich Pérez",
  "job_title": "Director General",
  "department": "Dirección General",
  "warehouse": "Cabo San Lucas",
  "hire_date_str": "2013-01-01"
},
{
  "id_odoo": 2,
  "name": "Ángela Aracely Costich Pérez",
  "job_title": "Administración General",
  "department": "Dirección General",
  "warehouse": "Cabo San Lucas",
  "hire_date_str": "2026-08-22"
}]

ingresados = []

for row in employees:
    x = requests.post(endpoint, json = row)
    if x.ok:
            ingresados.append(x.ok)
    else:
        print(x, ' para ', row['id_odoo'])
        print(x.json())

print(f'{len(ingresados)} de {len(employees)} empleados ingresados con éxito')

Ingresar empleados desde Odoo (tiempo_real)

In [ ]:
payload = {
        "domain": [("active", "!=", False)],
        "fields": [
            'id',
            'name',
            'job_title',
            'department_id',
            'x_hire_date',
            'x_warehouse_id',
        ],
        "context": {"lang": "es_MX"},
    }

empleados_data = api.search_read('hr.employee', payload=payload)

empleados_df = pd.DataFrame(empleados_data)

empleados_df.rename(columns={
    'id': 'id_odoo',
    'x_hire_date': 'hire_date_str',
}, inplace=True)

empleados_df['department'] = empleados_df['department_id'].apply(lambda x: 'Ventas Piso' if 'Ventas Piso' in x[1] else x[1])
empleados_df['warehouse'] = empleados_df['x_warehouse_id'].apply(lambda x: 'Cabo San Lucas' if x[0]==2 else 'San José Del Cabo')
empleados_df.drop(columns=['department_id', 'x_warehouse_id'], inplace=True)

endpoint = url + '/employees/new'


ingresados = []
id_errores = []

for i, row in empleados_df.iterrows():
    empleado = row.fillna(None).to_dict()
    x = requests.post(endpoint, json = empleado)
    if x.ok:
            ingresados.append(x.ok)
    else:
        print(x, ' para ', row['id_odoo'])
        print(x.json())
        id_errores.append(row.name)

print(f'{len(ingresados)} de {len(empleados_df)} empleados ingresados con éxito')

Ingresar equipos desde archivo

In [ ]:
archivo_lp = '/equipos simulacion.xlsx'
eq_simulacion_df = pd.read_excel(str(ruta_carpeta)+archivo_lp)

endpoint = url + '/teams/new'

ingresados = []
id_errores = []

for i, row in eq_simulacion_df.iterrows():
    producto = row.fillna(None).to_dict()
    x = requests.post(endpoint, json = producto)
    if x.ok:
        ingresados.append(x.ok)
    else:
        print(x, ' para ', row['Eq'])
        print(x.json())
        id_errores.append(row.name)

print(f'{len(ingresados)} de {len(eq_simulacion_df)} equipos ingresados con éxito')

Ingresar productos desde archivo lp_simulación

In [ ]:
archivo_lp = '/Lista de Productos 2025 A1 simulacion.xlsx'
lp_simulacion_df = pd.read_excel(str(ruta_carpeta)+archivo_lp)
lp_simulacion_df['code'] = lp_simulacion_df['code'].astype(str)

endpoint = url + '/products/new'

ingresados = []
id_errores = []

for i, row in lp_simulacion_df.iterrows():
    producto = row.fillna(None).to_dict()
    x = requests.post(endpoint, json = producto)
    if x.ok:
        ingresados.append(x.ok)
    else:
        print(x, ' para ', row['id_odoo'])
        print(x.json())
        id_errores.append(row.name)

print(f'{len(ingresados)} de {len(lp_simulacion_df)} productos ingresados con éxito')

Ingresar conteos individuales desde archivos

In [ ]:
df_prod = pd.read_sql_table('products', con=engine, columns=['id','code'])
df_prod['code'] = df_prod['code'].astype(str)
df_prod.rename(columns={'id': 'id_products'}, inplace=True)

df_eq = pd.read_sql_table('teams', con=engine)

archivo_lp = '/equipos simulacion.xlsx'
eq_simulacion_df = pd.read_excel(str(ruta_carpeta)+archivo_lp)

nombres = []
for i in range(len(eq_simulacion_df)):
    row = eq_simulacion_df.iloc[i]
    nombres.append([row['a'], row['empleado_id_a']])
    nombres.append([row['b'], row['empleado_id_b']])


ruta_archivos_conteos = ruta_carpeta.joinpath('Archivos de conteo')

for nombre in nombres:
    for p in ruta_archivos_conteos.iterdir():
        if nombre[0] == p.stem[:-7]:
            print(f'Comiensa {nombre[0]}, con id {nombre[1]}')
            nombre.append(True)
            df = pd.read_excel(str(p))
            df.rename(columns={'Código': 'code', nombre[0]: 'count_qty'}, inplace=True)
            df.dropna(subset=[df.keys()[2]], inplace=True)
            df['code'] = df['code'].astype(str)
            df['assigned_team_id'] = df_eq.loc[(df_eq['empleado_id_a'].isin([nombre[1]])) | (df_eq['empleado_id_b'].isin([nombre[1]])), 'id'].iloc[0]
            df['assigned_user_id'] = nombre[1]
            df['counting_user_id'] = nombre[1]
            df['count_type'] = 'individual'
            df.reset_index(drop=True, inplace=True)
            
            dfmerge = pd.merge(df, df_prod, on='code', how='left')
        
            if dfmerge.loc[dfmerge['id_products'].isna()].empty:
                print('Se encontraron todos los códigos, se procede a registrar en el endpoint')

                endpoint = url + '/inventory/registers/new'

                ingresados = []
                id_errores = []

                for i, row in dfmerge.iterrows():
                    conteo = row.fillna(None).to_dict()
                    x = requests.post(endpoint, json = conteo)
                    if x.ok:
                        ingresados.append(x.ok)
                    else:
                        print(x, ' para ', row['id_products'])
                        print(x.json())
                        id_errores.append(row.name)

                print(f'{len(ingresados)} de {len(dfmerge)} conteos ingresados con éxito')
                print('\n')

            else:
                print(f'Hay errores con {nombre[0]}')
                print('\n')

todos_con_conteo = []
for nombre in nombres:
    if nombre[2]:
        todos_con_conteo.append(True)
    else:
        todos_con_conteo.append(False)
        print(f'{nombre[0]} no tiene archivo de conteo, faltan sus registros')

if all(todos_con_conteo):
    print('\n\nTodos los usuarios fueron mapeados con un archivo de conteo')